# nanoGenRec Public MovieLens GPU Benchmark

This notebook runs the redistributable nanoGenRec public benchmark on Google Colab GPU. It follows the strict framework loop: public MovieLens ratings/metadata -> Qwen3 item embeddings -> CPU KMeans Semantic IDs -> tiny NTP training -> GRPO-style reward alignment -> SID-constrained full-recall evaluation.

Before running: choose `Runtime` -> `Change runtime type` -> `T4 GPU`.


In [ ]:
!nvidia-smi

## Clone nanoGenRec

Re-running this cell updates an existing checkout.

In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path("/content/nanoGenRec")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/yzq986/nanoGenRec.git", str(repo)], check=True)
os.chdir(repo)
print(f"Working directory: {Path.cwd()}")


## Install Dependencies

Qwen3 embedding requires a recent Transformers stack.


In [ ]:
!python -m pip install -q -U "transformers>=4.51.0" "tokenizers>=0.21.0" "accelerate>=0.25.0" safetensors


## Recommended T4 Run

This is the recommended strict public run. It uses Qwen3 embeddings and a small GRPO-style reward-alignment stage before evaluation.


In [ ]:
import json
import subprocess
import time
from pathlib import Path

output_dir = Path("public_benchmarks/runs/ml-latest-small-qwen-rl-t4")
cmd = [
    "python", "run.py", "public-movielens",
    "--dataset", "ml-latest-small",
    "--output_dir", str(output_dir),
    "--min_rating", "5.0",
    "--min_user_items", "5",
    "--max_users", "0",
    "--max_items_per_user", "100",
    "--feature_source", "qwen",
    "--qwen_device", "cuda",
    "--qwen_batch_size", "16",
    "--clusters", "16,16,16",
    "--kmeans_iters", "5",
    "--kmeans_sample_size", "2048",
    "--train_mode", "sliding",
    "--min_context_items", "2",
    "--embed_dim", "96",
    "--n_heads", "4",
    "--layers", "2",
    "--batch_size", "128",
    "--epochs", "8",
    "--rl_steps", "50",
    "--rl_batch_size", "8",
    "--rl_group_size", "8",
    "--eval_samples", "1000",
    "--beam_size", "1000",
    "--max_seq_len", "128",
    "--device", "cuda",
]
start = time.time()
subprocess.run(cmd, check=True)
elapsed = time.time() - start
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
    text=True,
    capture_output=True,
    check=False,
)
runtime = {"elapsed_seconds": elapsed, "gpu": gpu.stdout.strip(), "command": cmd}
(output_dir / "runtime.json").write_text(json.dumps(runtime, indent=2))
print(json.dumps(runtime, indent=2))


## Inspect metrics

In [ ]:
import json
from pathlib import Path

run_dir = Path("public_benchmarks/runs/ml-latest-small-qwen-rl-t4")
metrics = json.loads((run_dir / "metrics.json").read_text())
print(json.dumps(metrics, indent=2))
runtime_path = run_dir / "runtime.json"
if runtime_path.exists():
    print(json.dumps(json.loads(runtime_path.read_text()), indent=2))


## Optional: Larger Run

If the T4 session is stable, try MovieLens 1M. This is the next useful public scale. If Colab disconnects, reduce `--epochs`, `--beam_size`, `--eval_samples`, or `--rl_steps` first.


In [ ]:
# !python run.py public-movielens \
#     --dataset ml-1m \
#     --output_dir public_benchmarks/runs/ml-1m-qwen-rl-t4 \
#     --min_rating 4.0 \
#     --min_user_items 10 \
#     --max_users 0 \
#     --max_items_per_user 100 \
#     --feature_source qwen \
#     --qwen_device cuda \
#     --qwen_batch_size 16 \
#     --clusters 64,64,64 \
#     --kmeans_iters 5 \
#     --kmeans_sample_size 8192 \
#     --train_mode sliding \
#     --min_context_items 2 \
#     --embed_dim 128 \
#     --n_heads 4 \
#     --layers 3 \
#     --batch_size 256 \
#     --epochs 5 \
#     --rl_steps 100 \
#     --rl_batch_size 8 \
#     --rl_group_size 8 \
#     --eval_samples 1000 \
#     --beam_size 1000 \
#     --max_seq_len 128 \
#     --device cuda


## Optional: save outputs to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/nanoGenRec-public
# !cp -r public_benchmarks/runs/ml-latest-small-qwen-rl-t4 /content/drive/MyDrive/nanoGenRec-public/
